# Task Optimization: Dynamic Programming vs Greedy Algorithm

In [7]:
import numpy as np
import pandas as pd



## Version 2:
We will use duration instead of a deadline and sliding scale (1-5) for priority. Since it does not have deadline, the main pseudocode will change. It will also allow us to create a 2D knapsack which allows for 2 constraints instead of one. The two constraints will be the task's duration and cognitive capacity. This is more for task selection without the added process of scheduling. In implementing, I did confuse both scheduling and task selection and that does change the idea. So I will create two versions and see which one I prefer and finalize as I start my analysis.

## Build Knapsack array

```
Input: Input: Tasks, Capacity, Total time
Goal: The goal is to build a double-constrained knapsack array

* W_C = Cognitive Capacity
* W_T = Total Time
* Values = Array of priority per task
* Weights_cog = Array of cognitive cost per task
* Weights_time = Array of duration per task
```

In [1]:
def build_knapsack_array(tasks, capacity, time_budget):

    # INITIALIZATION
    weights_cog = tasks.cognitive_cost.tolist()
    weights_duration = tasks.duration.tolist()
    values = tasks.priority.tolist()

    # Create a 3D array with number of tasks x cognitive capacity x total time and feel with 0s
    knapsack_array = np.zeros((len(tasks)+1, time_budget+1, capacity+1), dtype = int)

    # Go through each row which represents one task
    for r in range(1, len(tasks)+1):
        for c_time in range(1, time_budget+1):
            for c_cap in range(1, capacity+1):

                # Cognitive cost
                task_cog = weights_cog[r-1]
                # duration cost
                task_duration = weights_duration[r-1]

                # if the cost of the task is grater than the capacity or duration represented by the column, we skip it
                if task_duration > c_time or task_cog > c_cap:
                    # We will use the previous score of the row above the column
                    knapsack_array[r][c_time][c_cap] = knapsack_array[r-1][c_time][c_cap]

                # On the other hand, if the capacity cost of the task is <= the capacity represented by the column, we are tasked to see which score is more, that if we skip it or that if we keep it
                else:
                    # First we will calculate if we keep the score, we want to remove the cost from the max capacity
                    time_weight_diff = c_time - task_duration
                    capacity_weight_diff = c_cap - task_cog

                    # the score is calculated by adding the priority of the task to the remaining score... to get the remaining score, you look one row above, and you choose the column based on the weight difference we observed previously.
                    kept_score = values[r-1] + knapsack_array[r-1][time_weight_diff][capacity_weight_diff]

                    # The final score takes the max of the kept score and the previous score which is the row above (same column)
                    final_score = max(knapsack_array[r-1][c_time][c_cap], kept_score)
                    knapsack_array[r][c_time][c_cap] = final_score

    return knapsack_array

## Knapsack Traceback

```
Input: Knapsack Array, Tasks
Goal: To return scheduled and unscheduled tasks after tracing the knapsack array
```

In [2]:
def knapsack_traceback(knapsack_array, tasks):
    scheduled = []
    unscheduled = tasks.task_name.tolist()
    duration_list = tasks.duration.tolist()
    capacity_list = tasks.cognitive_cost.tolist()

    # We want to start at the last cell for both time anc capacity
    c_time = knapsack_array.shape[1]-1
    c_cap = knapsack_array.shape[2]-1

    # Starting at the last row and last column, decrement through each row
    for r in range(len(tasks), 0, -1):
        # Compare the current row score to the score above (row - 1, column is same)
        if knapsack_array[r][c_time][c_cap] != knapsack_array[r-1][c_time][c_cap]:
            # if it is different that means the row was added to the knapsack so we want to append it to our task list
            scheduled.append(unscheduled[r-1])
            # Remove the task from the unscheduled list
            unscheduled.remove(unscheduled[r-1])
            # The capacity decreases by the weight of the task according the current row
            c_time = c_time - duration_list[r-1]
            c_cap = c_cap - capacity_list[r-1]

    # Return scheduled and unscheduled to-do lists
    return scheduled, unscheduled


## Knapsack Driver

```
Input: Leftover Tasks, Remaining Capacity
Goal: Find the optimal combination of tasks is within the remaining cognitive capacity while maximizing total priority

* array = Build knapsack array(tasks, W)
* scheduled, unscheduled = traceback(array, tasks)
* Return scheduled and unscheduled list
```


In [3]:
def knapsack(tasks, capacity, time_budget):
    array = build_knapsack_array(tasks, capacity, time_budget)
    scheduled, unscheduled = knapsack_traceback(array, tasks)
    return scheduled, unscheduled

## Greedy

```
Input: Tasks, Total Time
Goal: To maximize priority greedily wihthin the remaining cognitive capacity

* Sort Leftover tasks by priority descending
* Then sort leftover tasks by (deadline + cost) ascending
* Initialize current load to 0
* Initialize empty scheduled list
* Initialize tasks to unscheduled list

* for each task in sorted tasks:
   * task weight = deadline + cose
   if current_load + task weight <= remaining capacity
      * append task to scheduled list
      * remove task from unscheduled list
      * Add task weight to current load

* return scheduled and unscheduled lists
```

In [10]:
def greedy(tasks,time_budget, greedy_type = None):
    if greedy_type is None or greedy_type == 0:
        # Sort tasks by priority descending, then duration ascending (Tiebreaker: If two tasks have the same priority, prefer the shortest duration one)
        tasks_sorted = tasks.sort_values(by=["priority", "duration"], ascending=[False, True])
    elif greedy_type == 1:
        # Sort tasks by duration ascending, then priority descending (Tiebreaker: If two tasks have the same duration, prefer the highest priority one)
        tasks_sorted = tasks.sort_values(by=["duration", "priority"], ascending=[True, False])
    elif greedy_type == 2:
        # Add column for highest value per time
        tasks["ratio"] = tasks["priority"] / tasks["duration"]
        # Sort tasks by ratio descending, then priority descending (Tiebreaker: If two tasks have same efficiency score, prefer highest priority one)
        tasks_sorted = tasks.sort_values(by=["ratio", "priority"], ascending=[False, False])


    # initial current time spent to 0
    time_spent = 0

    # Initialize empty scheduled list
    scheduled = []

    # Initialize tasks to unscheduled list
    unscheduled = tasks.task_name.tolist()

    # for each task in sorted list
    for _,task in tasks_sorted.iterrows():
        # if cumulative time spent <= time budgeted
        if time_spent + task.duration <= time_budget:
            # Append the task to schedule
            scheduled.append(task.task_name)
            # Remove task from unscheduled list
            unscheduled.remove(task.task_name)
            # Add task duration to time spent
            time_spent += task.duration
    return scheduled, unscheduled


## Main Function

```
Input CSV file

* Read file and parse into a dataframe
* Validate no fields are missing as per assumption


* Check if len(df) > 0
   * dp_scheduled, dp_unscheduled = Knapsack(ldf, capacity, total_time)
   * g_scheduled, d_unscheduled = Greedy(df, capacity, total_time)

   * print(dp_scheduled)
   * print(g_scheduled)

* else print "No tasks for today"
```

In [21]:
def task_optimizer(file, cog_capacity = 80, time_budget = 240):
    greedy_type = {"Priority_First":0, "Duration_First": 1, "Highest_Value_Per_time": 2}
    # Read in data
    with open(file, mode='r', newline='') as csv_file:
        df = pd.read_csv(csv_file)
        print(f"Tasks List: {df}")
        results = {
        "knapsack": {},
        "greedy_priority": {},
        "greedy_duration": {},
        "greedy_ratio": {},
    }

    if len(df) > 0:
        results["knapsack"]["scheduled"], results["knapsack"]["unscheduled"] = knapsack(df, cog_capacity, time_budget)
        results["greedy_priority"]["scheduled"], results["greedy_priority"]["unscheduled"] = greedy(df, time_budget,
                                                                                                    greedy_type["Priority_First"])
        results["greedy_duration"]["scheduled"], results["greedy_duration"]["unscheduled"] = greedy(df, time_budget,
                                                                                                    greedy_type[
                                                                                                        "Duration_First"])
        results["greedy_ratio"]["scheduled"], results["greedy_ratio"]["unscheduled"] = greedy(df, time_budget,
                                                                                              greedy_type[
                                                                                                  "Highest_Value_Per_time"])
    else:
        print("No tasks for today!")

    return task_df, results



In [22]:
task_df, results = task_optimizer(file = "../data/Version2_duration.csv")

Tasks List:                  task_name  duration  cognitive_cost  priority
0              Run 2 miles        45              20         7
1    Complete Peer Reviews        60               5         6
2           Complete Quiz         30               5         6
3   Read Literature Review       120              25         7
4                  Pilates        60              20        10
5         Meal Prep Dinner        60              10         5
6  Wash 2 Loads of Laundry        30               5         3
7        Complete Duolingo         5               1        10
8   Book Flights to Boston        10               1         6
9     Pick up Dry-Cleaning        15               3         1


## Task List

In [37]:
def print_task_list(scheduled, unscheduled, algorithm_implemented):
    # Using ANSI escape codes to cross out the tasks that were not scheduled
    strikethrough_start = "\x1b[9m"
    reset = "\x1b[0m"

    print(f"\033[1mTask List for {algorithm_implemented}: \033[0m\n")
    task_list = []
    # Task List
    print(f"Task-List:\n")
    for task in scheduled:
        task_list.append(scheduled)
        print(f"* {task}")
    for task in unscheduled:
        print(f"* {strikethrough_start}{task}{reset}")


## Score Card

In [34]:
def score_card(task_df, scheduled):

    total_priority = 0
    total_duration = 0
    total_cog_cost = 0

    for task in scheduled:
        total_priority += task_df.loc[task_df["task_name"] == task, 'priority'].values[0]
        total_duration += task_df.loc[task_df["task_name"] == task, 'duration'].values[0]
        total_cog_cost += task_df.loc[task_df["task_name"] == task, 'cognitive_cost'].values[0]

    score_card = pd.DataFrame()
    score_card["Total Priority"] = [total_priority]
    score_card["Total Duration (min)"] = [total_duration]
    score_card["Total Capacity"] = [total_cog_cost]
    score_card["Total Tasks Scheduled"] = [len(scheduled)]

    print(f"\n{score_card}\n")

    return



In [38]:
print_task_list(results["knapsack"]["scheduled"], results["knapsack"]["unscheduled"], "Knapsack Algorithm")
print(score_card(task_df, results["knapsack"]["scheduled"]))

Task List for Knapsack Algorithm: 

Task-List:

* Book Flights to Boston
* Complete Duolingo
* Wash 2 Loads of Laundry
* Pilates
* Complete Quiz 
* Complete Peer Reviews
* Run 2 miles
* Read Literature Review
* Meal Prep Dinner
* Pick up Dry-Cleaning

   Total Priority  Total Duration (min)  Total Capacity  Total Tasks Scheduled
0              48                   240              57                      7

None


In [39]:
print_task_list(results["greedy_priority"]["scheduled"], results["greedy_priority"]["unscheduled"], "Greedy Algorithm (Highest Priority)")
print(score_card(task_df, results["greedy_priority"]["scheduled"]))

Task List for Greedy Algorithm (Highest Priority): 

Task-List:

* Complete Duolingo
* Pilates
* Run 2 miles
* Read Literature Review
* Book Flights to Boston
* Complete Peer Reviews
* Complete Quiz 
* Meal Prep Dinner
* Wash 2 Loads of Laundry
* Pick up Dry-Cleaning

   Total Priority  Total Duration (min)  Total Capacity  Total Tasks Scheduled
0              40                   240              67                      5

None


In [40]:
print_task_list(results["greedy_duration"]["scheduled"], results["greedy_duration"]["unscheduled"], "Greedy Algorithm (Shortest Duration)")
print(score_card(task_df, results["greedy_duration"]["scheduled"]))

Task List for Greedy Algorithm (Shortest Duration): 

Task-List:

* Complete Duolingo
* Book Flights to Boston
* Pick up Dry-Cleaning
* Complete Quiz 
* Wash 2 Loads of Laundry
* Run 2 miles
* Pilates
* Complete Peer Reviews
* Read Literature Review
* Meal Prep Dinner

   Total Priority  Total Duration (min)  Total Capacity  Total Tasks Scheduled
0              43                   195              55                      7

None


In [41]:
print_task_list(results["greedy_ratio"]["scheduled"], results["greedy_ratio"]["unscheduled"], "Greedy Algorithm (Highest Value Per Time)")
print(score_card(task_df, results["greedy_ratio"]["scheduled"]))

Task List for Greedy Algorithm (Highest Value Per Time): 

Task-List:

* Complete Duolingo
* Book Flights to Boston
* Complete Quiz 
* Pilates
* Run 2 miles
* Complete Peer Reviews
* Wash 2 Loads of Laundry
* Read Literature Review
* Meal Prep Dinner
* Pick up Dry-Cleaning

   Total Priority  Total Duration (min)  Total Capacity  Total Tasks Scheduled
0              48                   240              57                      7

None


In [86]:
def build_score_card(task_df, results):
    score_card = pd.DataFrame()
    total_priority_per_alg = []
    total_duration_per_alg = []
    total_cog_cost_per_alg = []
    n_scheduled_per_alg = []

    for algorithm in results.keys():
        total_priority = 0
        total_duration = 0
        total_cog_cost = 0

        for task in results[algorithm]["scheduled"]:
            total_priority += task_df.loc[task_df["task_name"] == task, 'priority'].values[0]
            total_duration += task_df.loc[task_df["task_name"] == task, 'duration'].values[0]
            total_cog_cost += task_df.loc[task_df["task_name"] == task, 'cognitive_cost'].values[0]
        total_priority_per_alg.append(total_priority)
        total_duration_per_alg.append(total_duration)
        total_cog_cost_per_alg.append(total_cog_cost)
        n_scheduled_per_alg.append(len(results[algorithm]["scheduled"]))

    score_card["Algorithm"] = results.keys()
    score_card["Total Priority"] = total_priority_per_alg
    score_card["Total Duration (min)"] = total_duration_per_alg
    score_card["Total Capacity"] = total_cog_cost_per_alg
    score_card["Total Tasks Scheduled"] = n_scheduled_per_alg

    return score_card


In [87]:
build_score_card(task_df, results)

,Algorithm,Total Priority,Total Duration (min),Total Capacity,Total Tasks Scheduled
0,knapsack,48,240,57,7
1,greedy_priority,40,240,67,5
2,greedy_duration,43,195,55,7
3,greedy_ratio,48,240,57,7
